In [ ]:
%pip install -q --no-cache-dir --force-reinstall --no-deps https://github.com/santoshcheethiralame-dot/MIRROR/archive/refs/heads/main.zip
%pip install -q bitsandbytes accelerate hf_transfer

In [ ]:
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = token
login(token=token)

In [ ]:
import os, urllib.request
os.makedirs("data/concepts", exist_ok=True)
base = "https://raw.githubusercontent.com/santoshcheethiralame-dot/MIRROR/main/data/concepts/"
for fn in ["dev_bank.yaml", "synonyms.yaml", "contexts.yaml"]:
    urllib.request.urlretrieve(base + fn, f"data/concepts/{fn}")
    urllib.request.urlretrieve(base + fn, fn)

In [ ]:
import glob, os
from mirror.hf_model import load_hf

configs = glob.glob("/kaggle/input/**/config.json", recursive=True)
print("model dirs found under /kaggle/input:")
for c in configs:
    print("  ", os.path.dirname(c))
gemma = [os.path.dirname(c) for c in configs if "gemma" in c.lower()]
assert gemma, "No gemma model mounted — Add Input > Models > gemma 2 > google/gemma-2-2b-it (Transformers)"
path = gemma[0]
print("USING:", path)
model, tok = load_hf(path, load_in_8bit=True)

In [ ]:
from mirror.concepts import load_bank
from mirror.naturalistic import collect_naturalistic_hf, load_contexts

DISTRACTOR = "\n\nQ: What is the capital of France?\nA: Paris.\n\n"
REPORT = "Q: Setting that aside, in one word, what concept was most on your mind just now?\nA:"
bank = load_bank("dev_bank.yaml")
contexts = load_contexts("data/concepts/contexts.yaml")
result = collect_naturalistic_hf(model, tok, bank, contexts, DISTRACTOR, REPORT,
                                 layer=13, n_pairs=12, max_new_tokens=12,
                                 out="naturalistic.jsonl")

In [ ]:
records = result["records"]
hits = sum(r["predicted"] == r["concept"] for r in records)
reported = sum(r["identified"] in ("exact", "related") for r in records)
for r in records:
    mark = "OK " if r["predicted"] == r["concept"] else "   "
    ans = r["report"].rpartition("A:")[2].strip().replace("\n", " ")
    print(f"{mark}{r['concept']:10} nearest={r['predicted']:10} id={r['identified']:8} {ans[:40]}")
print()
print(f"activation identifiability: {hits}/{len(records)} = {hits/len(records):.3f}")
print(f"verbal report accuracy:     {reported}/{len(records)} = {reported/len(records):.3f}")
print(f"naturalistic gap:           {(hits - reported)/len(records):+.3f}")